In [1]:
from pathlib import Path

from docling_graph.templategen import (
    DocumentContent,
    build_llm_call_fn,
    induce_spec_from_documents,
)

In [2]:
import os
from dotenv import load_dotenv

load_dotenv()

gemini_api_key = os.getenv("GEMINI_API_KEY")

In [3]:
yaml_text = Path("university.yaml").read_text(
    encoding="utf-8"
)

In [ ]:
# Gemini LLM
llm_call = build_llm_call_fn(
    "gemini",
    "gemini-2.5-flash",
    structured_output=False,
)



In [5]:
# Automatically infer schema from YAML
spec, report = induce_spec_from_documents(
    [
        DocumentContent(
            name="university_yaml",
            text=yaml_text
        )
    ],
    llm_call,
    strategy="one-shot",
)

In [6]:
print(spec.model_dump_json(indent=2))

{
  "module_docstring": "Knowledge-graph template induced from 1 sample document(s): university_yaml.",
  "root": "University",
  "enums": [],
  "models": [
    {
      "name": "University",
      "kind": "root",
      "docstring": "The main academic institution described in the document.",
      "identity_fields": [
        "identifier"
      ],
      "max_instances": null,
      "fields": [
        {
          "name": "identifier",
          "type": "str",
          "is_list": false,
          "description": "The unique identifier for the university.",
          "examples": [],
          "role": "identity",
          "edge_label": null,
          "reference": false,
          "closed_catalog": false,
          "normalizer": "none",
          "unit": null,
          "evidence": []
        },
        {
          "name": "name",
          "type": "str",
          "is_list": false,
          "description": "The full name of the university, found under 'university.name'.",
          "exam

# Using LLM for template

In [7]:
from docling_graph.templategen import (
    render_template,
    verify_template_source,
)

# Generate Pydantic template source in memory
template_source = render_template(spec)

# Verify it
verification = verify_template_source(
    template_source,
    root_class=spec.root,
    spec=spec,
)

print("Root class:", spec.root)
print("Template valid:", verification.passed)


# Execute generated template in memory
template_namespace = {}

exec(
    template_source,
    template_namespace
)

# Get root class dynamically
RootTemplate = template_namespace[spec.root]

print(
    "Template loaded in memory:",
    RootTemplate
)

Root class: University
Template valid: True
Template loaded in memory: <class 'University'>


# Graph generation with preprocessed Yaml Text

In [8]:
from docling_graph import (
    PipelineConfig,
    run_pipeline,
)

config = PipelineConfig(
    source=yaml_text,

    template=RootTemplate,

    backend="llm",
    inference="remote",

    provider_override="gemini",
    model_override="gemini-2.5-flash",

    processing_mode="many-to-one",
)

context = run_pipeline(
    config,
    mode="api"
)

graph = context.knowledge_graph

print(
    "Nodes:",
    graph.number_of_nodes()
)

print(
    "Edges:",
    graph.number_of_edges()
)

10:36:51 INFO     [Pipeline] Starting Docling-Graph Pipeline
10:36:51 INFO     [Input Normalization] Detecting input type (mode: api)...
10:36:51 INFO     [Input Normalization] Detected: document
10:36:51 INFO     [Input Normalization] Validating input...
10:36:51 INFO     [Input Normalization] Loading and normalizing input...
10:36:51 INFO     [Input Normalization] Normalized successfully
10:36:51 INFO     [Input Normalization] Processing flags: skip_ocr=False, skip_segmentation=False
10:36:51 INFO     [Template Loading] Loading template...
10:36:51 INFO     [Template Loading] Loaded: University
10:36:51 INFO     [Extraction] Creating extractor...
10:36:51 INFO     [Extraction] Using model: gemini-2.5-flash (provider: gemini)
10:36:51 INFO     [litellm] LiteLLMClient initialized for model: gemini-2.5-flash
10:36:51 INFO     [LlmBackend] Initialized (client=LiteLLMClient, model=gemini-2.5-flash)
10:36:52 WARNING  [huggingface_hub] Warning: You are sending unauthenticated requests to th

Nodes: 7
Edges: 6


In [9]:
print("\n--- NODES ---")

for node_id, data in graph.nodes(data=True):
    print(node_id, data)


print("\n--- EDGES ---")

for source, target, data in graph.edges(data=True):
    print(source, "->", target, data)


--- NODES ---
Professor_c7bd4d76cc7e4db8 {'id': 'Professor_c7bd4d76cc7e4db8', 'label': 'Professor', 'type': 'entity', '__class__': 'Professor', 'identifier': 'professor_001', 'name': None, 'title': None, 'research_areas': [], '__provenance__': {'document_id': '065c1ae8a03c63716656388c1e278410', 'match': 'verbatim', 'chunks': [0], 'pages': [], 'refs': ['#/texts/0', '#/texts/1', '#/texts/2', '#/texts/3', '#/texts/4', '#/texts/5', '#/texts/6', '#/texts/7']}}
Student_96481115c02b525f {'id': 'Student_96481115c02b525f', 'label': 'Student', 'type': 'entity', '__class__': 'Student', 'identifier': 'student_001', 'name': None, 'program': None, 'skills': [], '__provenance__': {'document_id': '065c1ae8a03c63716656388c1e278410', 'match': 'verbatim', 'chunks': [0], 'pages': [], 'refs': ['#/texts/0', '#/texts/1', '#/texts/2', '#/texts/3', '#/texts/4', '#/texts/5', '#/texts/6', '#/texts/7']}}
Student_a4b393471f6b428d {'id': 'Student_a4b393471f6b428d', 'label': 'Student', 'type': 'entity', '__class__'

In [10]:
import os

import psycopg
from dotenv import load_dotenv

load_dotenv()

conn = psycopg.connect(
    host=os.getenv("PG_HOST"),
    port=os.getenv("PG_PORT"),
    dbname=os.getenv("PG_DATABASE"),
    user=os.getenv("PG_USER"),
    password=os.getenv("PG_PASSWORD"),
)

# Test connection by uploading one test node in psql

In [11]:
with conn.cursor() as cursor:

    # AGE must be loaded for this PostgreSQL session
    cursor.execute("LOAD 'age';")


    cursor.execute(
        'SET search_path = ag_catalog, "$user", public;'
    )

    cursor.execute(
        """
        SELECT *
        FROM cypher('yaml_graph', $$
            MATCH (n:TestNode)
            RETURN n
        $$) AS (n agtype);
        """
    )

    rows = cursor.fetchall()

    for row in rows:
        print(row)


('{"id": 844424930131969, "label": "TestNode", "properties": {"name": "AGE test"}}::vertex',)


# Age Initilization

In [12]:
with conn.cursor() as cursor:
    cursor.execute("LOAD 'age';")
    cursor.execute(
        'SET search_path = ag_catalog, "$user", public;'
    )

print("AGE initialized")

AGE initialized


## Insert Docling Nodes into Apache AGE

This step takes the nodes generated by Docling Graph and stores them as vertices in the `yaml_graph` graph inside PostgreSQL using Apache AGE.

The code:
- Loads Apache AGE for the PostgreSQL session.
- Iterates through all nodes in the Docling-generated graph.
- Keeps `docling_id` so nodes can be matched later when creating relationships.
- Removes internal Docling metadata such as `__class__` and `__provenance__`.
- Converts Python values into valid Cypher values.
- Creates AGE vertices using Cypher `CREATE`.

Example:

```cypher
CREATE (n:Person {
    docling_id: 'Person_...',
    person_id: 'person_001',
    name: 'Alice Johnson',
    role: 'Engineering Manager'
})

# Graph DB Deletion

In [14]:
GRAPH_NAME = "university_graph"

conn.rollback()  # clears any previously failed transaction

with conn.cursor() as cursor:
    cursor.execute("LOAD 'age';")
    cursor.execute(
        'SET search_path = ag_catalog, "$user", public;'
    )

    cursor.execute(
        "SELECT 1 FROM ag_catalog.ag_graph WHERE name = %s;",
        (GRAPH_NAME,),
    )

    if cursor.fetchone() is not None:
        cursor.execute(
            "SELECT drop_graph(%s, true);",
            (GRAPH_NAME,),
        )
        print(f"Deleted graph: {GRAPH_NAME}")
    else:
        print(f"Graph does not exist: {GRAPH_NAME}")

conn.commit()

Deleted graph: university_graph


In [15]:
import re

GRAPH_NAME = "university_graph"

def validate_graph_name(name):
    if not re.fullmatch(
        r"[A-Za-z_][A-Za-z0-9_]*",
        name
    ):
        raise ValueError(
            f"Invalid graph name: {name}"
        )

    return name

graph_name = validate_graph_name(
    GRAPH_NAME
)

with conn.cursor() as cursor:

    cursor.execute("LOAD 'age';")

    cursor.execute(
        'SET search_path = ag_catalog, "$user", public;'
    )

    cursor.execute(
        "SELECT create_graph(%s);",
        (graph_name,)
    )

conn.commit()

print(
    "Graph created:",
    graph_name
)

Graph created: university_graph


In [16]:
print("Docling nodes:", graph.number_of_nodes())

Docling nodes: 7


In [17]:
def cypher_value(value):

    if value is None:
        return "null"

    if isinstance(value, bool):
        return "true" if value else "false"

    if isinstance(value, (int, float)):
        return str(value)

    if isinstance(value, list):
        return "[" + ", ".join(
            cypher_value(v)
            for v in value
        ) + "]"

    text = str(value)

    text = text.replace(
        "\\",
        "\\\\"
    )

    text = text.replace(
        "'",
        "\\'"
    )

    return f"'{text}'"

# Inserting docling nodes to psql

In [18]:
with conn.cursor() as cursor:

    cursor.execute("LOAD 'age';")

    cursor.execute(
        'SET search_path = ag_catalog, "$user", public;'
    )

    for node_id, data in graph.nodes(
        data=True
    ):

        label = validate_graph_name(
            data["label"]
        )

        properties = {
            "docling_id": node_id
        }

        ignored_fields = {
            "id",
            "label",
            "type",
            "__class__",
            "__provenance__",
        }

        for key, value in data.items():

            if key in ignored_fields:
                continue

            if value is None:
                continue

            properties[key] = value

        property_text = ", ".join(
            f"{key}: {cypher_value(value)}"
            for key, value
            in properties.items()
        )

        query = f"""
        SELECT *
        FROM cypher('{GRAPH_NAME}', $$
            CREATE (n:{label} {{
                {property_text}
            }})
            RETURN n
        $$) AS (n agtype);
        """

        cursor.execute(query)

        cursor.fetchone()

        print(
            "Created node:",
            label,
            node_id
        )

conn.commit()

print("All nodes committed")

Created node: Professor Professor_c7bd4d76cc7e4db8
Created node: Student Student_96481115c02b525f
Created node: Student Student_a4b393471f6b428d
Created node: Department Department_0d37daebae8eac86
Created node: Course Course_538042b785d99975
Created node: Course Course_dd630c5b6c84ed79
Created node: University University_f22e5b183513ac09
All nodes committed


# Inserting edges in psql

In [19]:
def validate_edge_label(label):
    if not re.fullmatch(
        r"[A-Za-z_][A-Za-z0-9_]*",
        label
    ):
        raise ValueError(
            f"Invalid edge label: {label}"
        )

    return label

In [20]:
with conn.cursor() as cursor:

    cursor.execute("LOAD 'age';")

    cursor.execute(
        'SET search_path = ag_catalog, "$user", public;'
    )

    for source, target, data in graph.edges(data=True):

        edge_label = validate_edge_label(
            data["label"]
        )

        query = f"""
        SELECT *
        FROM cypher('{GRAPH_NAME}', $$
            MATCH
                (a {{docling_id: '{source}'}}),
                (b {{docling_id: '{target}'}})
            CREATE
                (a)-[r:{edge_label}]->(b)
            RETURN r
        $$) AS (r agtype);
        """

        cursor.execute(query)

        cursor.fetchone()

        print(
            "Inserted edge:",
            source,
            f"-[{edge_label}]->",
            target
        )

conn.commit()

print("All edges committed")

Inserted edge: Department_0d37daebae8eac86 -[HAS_HEAD]-> Professor_c7bd4d76cc7e4db8
Inserted edge: Department_0d37daebae8eac86 -[HAS_STUDENT]-> Student_96481115c02b525f
Inserted edge: Department_0d37daebae8eac86 -[HAS_STUDENT]-> Student_a4b393471f6b428d
Inserted edge: University_f22e5b183513ac09 -[HAS_DEPARTMENT]-> Department_0d37daebae8eac86
Inserted edge: University_f22e5b183513ac09 -[OFFERS_COURSE]-> Course_538042b785d99975
Inserted edge: University_f22e5b183513ac09 -[OFFERS_COURSE]-> Course_dd630c5b6c84ed79
All edges committed


# Inserting Nodes and Edges in Neo4j Aura DB

In [21]:
! pip install neo4j


[notice] A new release of pip is available: 26.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [22]:
import os

from dotenv import load_dotenv
from neo4j import GraphDatabase

load_dotenv()

driver = GraphDatabase.driver(
    os.getenv("NEO4J_URI"),
    auth=(
        os.getenv("NEO4J_USERNAME"),
        os.getenv("NEO4J_PASSWORD"),
    ),
)

driver.verify_connectivity()

print("Connected to Neo4j")

Connected to Neo4j


In [23]:
with driver.session(database="system") as session:
    result = session.run("SHOW DATABASES")
    for record in result:
        print(record["name"])

neo4j
system


In [24]:
NEO4J_DATABASE = "neo4j"
GRAPH_NAME = "university_graph"

# Insert node in Neo4j DB

In [25]:
import re


def validate_label(label):
    if not re.fullmatch(
        r"[A-Za-z_][A-Za-z0-9_]*",
        label
    ):
        raise ValueError(
            f"Invalid Neo4j label: {label}"
        )

    return label


with driver.session(
    database=NEO4J_DATABASE
) as session:

    for node_id, data in graph.nodes(
        data=True
    ):

        label = validate_label(
            data["label"]
        )

        # Only ignore generic Docling/internal fields
        ignored = {
            "id",
            "label",
            "type",
            "__class__",
            "__provenance__",
        }

        properties = {
            "graph_name": GRAPH_NAME,
            "docling_id": node_id,
        }

        for key, value in data.items():

            if key in ignored:
                continue

            # Relationship placeholder fields are usually None,
            # so they are skipped automatically
            if value is None:
                continue

            properties[key] = value

        query = f"""
        MERGE (n:{label} {{
            graph_name: $graph_name,
            docling_id: $docling_id
        }})
        SET n += $properties
        RETURN n
        """

        result = session.run(
            query,
            graph_name=GRAPH_NAME,
            docling_id=node_id,
            properties=properties,
        )

        result.consume()

        print(
            "Inserted node:",
            label,
            data.get("name"),
            node_id
        )

Inserted node: Professor None Professor_c7bd4d76cc7e4db8
Inserted node: Student None Student_96481115c02b525f
Inserted node: Student None Student_a4b393471f6b428d
Inserted node: Department None Department_0d37daebae8eac86
Inserted node: Course None Course_538042b785d99975
Inserted node: Course None Course_dd630c5b6c84ed79
Inserted node: University None University_f22e5b183513ac09


# Insert Edges in Neo4j

In [26]:
def validate_edge_label(label):
    if not re.fullmatch(
        r"[A-Za-z_][A-Za-z0-9_]*",
        label
    ):
        raise ValueError(
            f"Invalid Neo4j relationship type: {label}"
        )

    return label


with driver.session(
    database=NEO4J_DATABASE
) as session:

    for source, target, data in graph.edges(
        data=True
    ):

        edge_label = validate_edge_label(
            data["label"]
        )

        query = f"""
        MATCH
            (a {{
                graph_name: $graph_name,
                docling_id: $source
            }}),
            (b {{
                graph_name: $graph_name,
                docling_id: $target
            }})

        MERGE (a)-[r:{edge_label}]->(b)

        RETURN r
        """

        result = session.run(
            query,
            graph_name=GRAPH_NAME,
            source=source,
            target=target,
        )

        result.consume()

        print(
            "Inserted edge:",
            source,
            f"-[{edge_label}]->",
            target,
        )

Inserted edge: Department_0d37daebae8eac86 -[HAS_HEAD]-> Professor_c7bd4d76cc7e4db8
Inserted edge: Department_0d37daebae8eac86 -[HAS_STUDENT]-> Student_96481115c02b525f
Inserted edge: Department_0d37daebae8eac86 -[HAS_STUDENT]-> Student_a4b393471f6b428d
Inserted edge: University_f22e5b183513ac09 -[HAS_DEPARTMENT]-> Department_0d37daebae8eac86
Inserted edge: University_f22e5b183513ac09 -[OFFERS_COURSE]-> Course_538042b785d99975
Inserted edge: University_f22e5b183513ac09 -[OFFERS_COURSE]-> Course_dd630c5b6c84ed79


# Verify Nodes Count

In [27]:
with driver.session(database=NEO4J_DATABASE) as session:

    result = session.run(
        """
        MATCH (n {graph_name: $graph_name})
        RETURN count(n) AS node_count
        """,
        graph_name=GRAPH_NAME,
    )

    print(
        "Node count:",
        result.single()["node_count"]
    )

Node count: 7


# Verify Edge Count

In [28]:
with driver.session(database=NEO4J_DATABASE) as session:

    result = session.run(
        """
        MATCH
            (a {graph_name: $graph_name})
            -[r]->
            (b {graph_name: $graph_name})

        RETURN count(r) AS edge_count
        """,
        graph_name=GRAPH_NAME,
    )

    print(
        "Edge count:",
        result.single()["edge_count"]
    )

Edge count: 6


In [37]:
! pip install google-genai

  Using cached pyasn1_modules-0.4.2-py3-none-any.whl.metadata (3.5 kB)
  Using cached pycparser-3.0-py3-none-any.whl.metadata (8.2 kB)
   ---------------------------------------- 0.0/1.1 MB ? eta -:--:--
   ---------------------------------------- 1.1/1.1 MB 5.4 MB/s  0:00:00
   ---------------------------------------- 0.0/3.8 MB ? eta -:--:--
   -------- ------------------------------- 0.8/3.8 MB 5.4 MB/s eta 0:00:01
   ---------- ----------------------------- 1.0/3.8 MB 3.6 MB/s eta 0:00:01
   ------------------- -------------------- 1.8/3.8 MB 3.2 MB/s eta 0:00:01
   --------------------------- ------------ 2.6/3.8 MB 3.3 MB/s eta 0:00:01
   ----------------------------------- ---- 3.4/3.8 MB 3.5 MB/s eta 0:00:01
   ---------------------------------------- 3.8/3.8 MB 3.6 MB/s  0:00:01
Using cached pyasn1_modules-0.4.2-py3-none-any.whl (181 kB)
Using cached pycparser-3.0-py3-none-any.whl (48 kB)

   ---------------------------------------- 0/7 [pycparser]
   ----- -------------------


[notice] A new release of pip is available: 26.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [49]:
import os
from google import genai

client = genai.Client(
    api_key=os.getenv("GEMINI_API_KEY")
)

In [50]:
def get_graph_schema(
    driver,
    database,
    graph_name
):
    schema = {
        "nodes": {},
        "relationships": []
    }

    with driver.session(
        database=database
    ) as session:

        node_result = session.run(
            """
            MATCH (n {graph_name: $graph_name})
            UNWIND labels(n) AS label
            UNWIND keys(n) AS property
            RETURN
                label,
                collect(DISTINCT property) AS properties
            """,
            graph_name=graph_name,
        )

        for record in node_result:
            schema["nodes"][
                record["label"]
            ] = record["properties"]

        rel_result = session.run(
            """
            MATCH
                (a {graph_name: $graph_name})
                -[r]->
                (b {graph_name: $graph_name})
            RETURN DISTINCT
                labels(a) AS source,
                type(r) AS relationship,
                labels(b) AS target
            """,
            graph_name=graph_name,
        )

        for record in rel_result:
            schema["relationships"].append({
                "source": record["source"],
                "relationship": record["relationship"],
                "target": record["target"],
            })

    return schema


neo4j_schema = get_graph_schema(
    driver,
    NEO4J_DATABASE,
    GRAPH_NAME,
)

In [51]:
def build_schema_prompt(schema):

    lines = [
        "Node labels and properties:"
    ]

    for label, properties in (
        schema["nodes"].items()
    ):
        lines.append(
            f"\n{label}: "
            f"{', '.join(properties)}"
        )

    lines.append(
        "\nRelationships:"
    )

    for rel in schema["relationships"]:

        source = ", ".join(
            rel["source"]
        )

        target = ", ".join(
            rel["target"]
        )

        lines.append(
            f"({source})"
            f"-[:{rel['relationship']}]->"
            f"({target})"
        )

    return "\n".join(lines)


GRAPH_SCHEMA = build_schema_prompt(
    neo4j_schema
)

print(GRAPH_SCHEMA)

Node labels and properties:

Professor: graph_name, docling_id, identifier, research_areas

Student: docling_id, skills, identifier, graph_name

Department: graph_name, docling_id, identifier, professors

Course: graph_name, docling_id, identifier, enrolled_students, technologies

University: docling_id, identifier, graph_name

Relationships:
(Department)-[:HAS_STUDENT]->(Student)
(Department)-[:HAS_HEAD]->(Professor)
(University)-[:OFFERS_COURSE]->(Course)
(University)-[:HAS_DEPARTMENT]->(Department)


In [52]:
def natural_language_to_cypher(
    question,
    graph_schema,
    graph_name
):

    prompt = f"""
    You are a Neo4j Cypher query generator.

    Generate one READ-ONLY Cypher query
    for the user's question.

    Use only this schema:

    {graph_schema}

    Rules:

    - Do not invent labels, properties,
    or relationship types.
    - Query only nodes belonging to:
    graph_name = '{graph_name}'
    - Never generate CREATE, MERGE,
    DELETE, SET, REMOVE or DROP.
    - Return only Cypher.
    - Do not use markdown fences.

    Question:
    {question}
    """

    response = client.models.generate_content(
        model="gemini-2.5-flash",
        contents=prompt,
    )

    return response.text.strip()

In [53]:
question = (
    "Which students belong "
    "to each department?"
)

cypher_query = natural_language_to_cypher(
    question,
    GRAPH_SCHEMA,
    GRAPH_NAME,
)

print(cypher_query)

ClientError: 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-2.5-flash\nPlease retry in 41.577924917s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'quotaMetric': 'generativelanguage.googleapis.com/generate_content_free_tier_requests', 'quotaId': 'GenerateRequestsPerDayPerProjectPerModel-FreeTier', 'quotaDimensions': {'location': 'global', 'model': 'gemini-2.5-flash'},

In [54]:
def run_cypher_query(
    driver,
    database,
    cypher_query
):
    with driver.session(
        database=database
    ) as session:

        result = session.run(
            cypher_query
        )

        records = [
            record.data()
            for record in result
        ]

    return records

In [ ]:
def generate_natural_language_answer(question,query_result):

    prompt = f"""
    You are answering a question using
    data returned from a Neo4j graph database.

    User question:

    {question}

    Database result:

    {query_result}

    Instructions:

    - Answer only from the database result.
    - Do not invent information.
    - If no records are returned, say that
    no matching data was found.
    - Give a concise natural-language answer.
    """

    response = client.models.generate_content(
        model="gemini-2.5-flash",
        contents=prompt,
    )

    return response.text.strip()

In [45]:
def ask_graph(question):

    # 1. Natural language -> Cypher
    cypher_query = natural_language_to_cypher(
        question=question,
        graph_schema=GRAPH_SCHEMA,
        graph_name=GRAPH_NAME,
    )

    print(
        "Generated Cypher:\n",
        cypher_query
    )

    # 2. Run Cypher on Neo4j
    query_result = run_cypher_query(
        driver=driver,
        database=NEO4J_DATABASE,
        cypher_query=cypher_query,
    )

    print(
        "\nRaw result:\n",
        query_result
    )

    # 3. Neo4j result -> natural language
    answer = generate_natural_language_answer(
        question=question,
        query_result=query_result,
    )

    return answer

In [48]:
question = (
    "Which students belong "
    "to each department?"
)

answer = ask_graph(
    question
)

print(
    "\nAnswer:\n",
    answer
)

ClientError: 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-2.5-flash\nPlease retry in 54.535774029s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'quotaMetric': 'generativelanguage.googleapis.com/generate_content_free_tier_requests', 'quotaId': 'GenerateRequestsPerDayPerProjectPerModel-FreeTier', 'quotaDimensions': {'model': 'gemini-2.5-flash', 'location': 'global'},

### Cypher query for get all nodes

```
MATCH (n {graph_name: 'yaml_graph'})
RETURN n;
```

### Cypher query for get all edges


```
MATCH
    (a {graph_name: 'yaml_graph'})
    -[r]->
    (b {graph_name: 'yaml_graph'})
RETURN a, r, b;
```

